### Evaluate diffusion model

Now that we have trained all of our model, we want to know how our models perform on different type on diffusion model

In [28]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from collections import defaultdict
from typing import Counter
import pandas as pd

### Art dataset analysis

In [2]:
ART_MODEL_PATH = "../weights/art_model.h5"
art_model = tf.keras.models.load_model(ART_MODEL_PATH)

In [3]:
raw_root = '../data/art/fake'
test_root = '../dataset_test/art/fake'
output_dir = '.'
os.makedirs(output_dir, exist_ok=True)

In [39]:
test_files = set(os.listdir(test_root))
print(f"Test files: {len(test_files)}") 

Test files: 11893


In [20]:
counts = defaultdict(Counter)



for diffusion_model in os.listdir(raw_root):
    print(f"Processing model: {diffusion_model}")
    model_dir = os.path.join(raw_root, diffusion_model)
    if not os.path.isdir(model_dir):
        continue

    # find intersection of raw-generated files and test set
    raw_files = set(os.listdir(model_dir))
    intersect_files = sorted(raw_files & test_files)

    predictions = []
    print(f"Found {len(intersect_files)} files in intersection")
    for fname in intersect_files:
        try:
            img = Image.open(os.path.join(test_root, fname)).convert('RGB')
            arr = np.array(img) / 255.0
            arr = np.expand_dims(arr, 0)

            preds = art_model.predict(arr, verbose=0)
            if preds.shape[-1] == 1:
                pred = int(preds[0][0] > 0.5)
            else:
                pred = int(np.argmax(preds, axis=1)[0])

            counts[diffusion_model][pred] += 1

        except Exception as e:
            print(f"Error on {fname}: {e}")

Processing model: AI_LD_romanticism
Found 1156 files in intersection
Processing model: .DS_Store
Processing model: AI_DiffusionDB_small_2
Found 2597 files in intersection
Processing model: AI_SD_ukiyo-e
Found 92 files in intersection
Processing model: AI_LD_expressionism
Found 699 files in intersection
Processing model: AI_LD_post_impressionism
Found 1178 files in intersection
Processing model: AI_DiffusionDB_small_1
Found 1623 files in intersection
Processing model: AI_LD_baroque
Found 1225 files in intersection
Processing model: AI_SD_impressionism
Found 1156 files in intersection
Processing model: AI_LD_renaissance
Found 1254 files in intersection
Processing model: AI_LD_art_nouveau
Found 913 files in intersection


In [ ]:
models = sorted(counts.keys())
rows = []
for model in models:
    total = sum(counts[model].values())
    misclassified = counts[model].get(1, 0)
    rows.append(
        {'model': model, 'misclassified': misclassified, 'total': total, 'percentage': f"{misclassified / total * 100:.2f}%"})
    
# order by misclassified
rows.sort(key=lambda x: x['percentage'], reverse=True)

# add rank
for i, row in enumerate(rows):
    row['rank'] = i + 1

df = pd.DataFrame(rows)
df

,model,misclassified,total,percentage,rank
0,AI_LD_expressionism,1,699,0.14%,1
1,AI_DiffusionDB_small_1,2,1623,0.12%,2
2,AI_LD_art_nouveau,1,913,0.11%,3
3,AI_DiffusionDB_small_2,2,2597,0.08%,4
4,AI_LD_post_impressionism,1,1178,0.08%,5
5,AI_LD_baroque,0,1225,0.00%,6
6,AI_LD_renaissance,0,1254,0.00%,7
7,AI_LD_romanticism,0,1156,0.00%,8
8,AI_SD_impressionism,0,1156,0.00%,9
9,AI_SD_ukiyo-e,0,92,0.00%,10
